In [0]:
from pyspark.sql.functions import *
from pyspark.sql import DataFrame
from delta.tables import DeltaTable

###2 normalizar strings


In [0]:
def normalizar_strings(df: DataFrame):
    df = df.withColumns({
        "primeiro_nome": trim(initcap(col("primeiro_nome"))),
        "ultimo_nome": trim(initcap(col("ultimo_nome"))),
        "email": trim(lower(col("email"))),
        "telefone" : trim(col("telefone")),
        "bi": trim(col("bi")),
        "nif": trim(col("nif")),
        "morada": trim(col("morada")),
        "provincia": trim(col("provincia")),
        "municipio": trim(col("municipio")),
        "profissao": trim(initcap(col("profissao"))),
        "genero": trim(col("genero"))
    })
    return df

###3 Padronização 

In [0]:
def padronizar_dados(df: DataFrame):

    df = df.withColumns({

        "telefone": regexp_replace(
            regexp_replace(
                col("telefone"),
                r"^(?:\+244|00244)",
                ""
            ),
            r"\D",
            ""
        ),
        "estado_civil": trim(
            when(col("estado_civil") == "Casada", "Casado")
                .when(col("estado_civil") == "Solteira", "Solteiro")
                .when(col("estado_civil") == "Divorciada", "Divorciado")
                .when(col("estado_civil") == "Viúva", "Viúvo")
                .otherwise(col("estado_civil"))
        )

    })
    return df

###4 Tratar nullos

In [0]:

def salvar_rejeitados(df: DataFrame, tabela: str):
    try:
        if not spark.catalog.tableExists(tabela):
            df.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(tabela)
        else:
            deltatable = DeltaTable.forName(spark, tabela)

            (
                deltatable.alias("destino") \
                .merge(df.alias("origem"), "destino.id = origem.id") \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
            )
        return True
    except Exception as e:
        print( f"Falha ao salvar tabela {tabela}: {e}")
        return False

def preencher_nulos_opcionais(df: DataFrame):
    df = df.withColumns({
        "email": when(col("email").isNull(), lit("S/N")).otherwise(col("email")),
        "telefone": when(col("telefone").isNull(), lit("S/N")).otherwise(col("telefone")),
        "bi": when(col("bi").isNull(), lit("S/N")).otherwise(col("bi")),
        "nif": when(col("nif").isNull(), lit("S/N")).otherwise(col("nif")),
        "profissao": when(col("profissao").isNull(), lit("S/N")).otherwise(col("profissao")),
        "estado_civil": when(col("estado_civil").isNull(), lit("S/N")).otherwise(col("estado_civil")),
        "genero": when(col("genero").isNull(), lit("S/N")).otherwise(col("genero")),
        "morada": when(col("morada").isNull(), lit("S/N")).otherwise(col("morada"))
    })
    return df

def tratar_nullos(df: DataFrame):

    df = df.withColumns({
        "primeiro_nome": when(col("primeiro_nome") == "", None)
            .otherwise(col("primeiro_nome")),
        "ultimo_nome": when(col("ultimo_nome") == "", None)
            .otherwise(col("ultimo_nome")),
        "provincia": when(col("provincia") == "", None)
            .otherwise(col("provincia")),
        "municipio": when(col("municipio") == "", None)
            .otherwise(col("municipio")),
        "email": when(col("email") == "", None)
            .otherwise(col("email")),
        "telefone": when(col("telefone") == "", None)
            .otherwise(col("telefone")),
        "bi": when(col("bi") == "", None)
            .otherwise(col("bi")),
        "nif": when(col("nif") == "", None)
            .otherwise(col("nif"))
    })
    df_rejeitados = df.filter(
        col("primeiro_nome").isNull() |
        col("ultimo_nome").isNull() |
        col("provincia").isNull() |
        col("municipio").isNull() |
        col("data_nascimento").isNull() |
        col("data_criacao").isNull() |
        col("rendimento_mensal").isNull()
    )
    salvar_rejeitados(df_rejeitados, "dbw_banco_orion.silver.clientes_rejeitados")
    df_validos = df.filter(
        col("id").isNotNull() &
        col("primeiro_nome").isNotNull()&
        col("ultimo_nome").isNotNull()&
        col("provincia").isNotNull()&
        col("municipio").isNotNull() &
        col("data_nascimento").isNotNull() &
        col("data_criacao").isNotNull() &
        col("rendimento_mensal").isNotNull()
    )
    df_validos = preencher_nulos_opcionais(df_validos)
    print(f"Total {df_rejeitados.count() + df_validos.count()}")
    print(f"Rejeitados {df_rejeitados.count()}")
    print(f"Validos {df_validos.count()}") 
    return df_validos

###5 Tratar duplicados

In [0]:
def tratar_duplicados(df: DataFrame):

    # =========================
    # BI duplicado
    # =========================
    df_bi_duplicados = (
        df.filter(
            col("bi").isNotNull() &
            (col("bi") != "S/N")
        )
        .groupBy("bi")
        .count()
        .filter(col("count") > 1)
        .select("bi")
    )

    # =========================
    # NIF duplicado
    # =========================
    df_nif_duplicados = (
        df.filter(
            col("nif").isNotNull() &
            (col("nif") != "S/N")
        )
        .groupBy("nif")
        .count()
        .filter(col("count") > 1)
        .select("nif")
    )

    # =========================
    # Email duplicado
    # =========================
    df_email_duplicados = (
        df.filter(
            col("email").isNotNull() &
            (col("email") != "S/N")
        )
        .groupBy("email")
        .count()
        .filter(col("count") > 1)
        .select("email")
    )

    # =========================
    # Registros com BI duplicado
    # =========================
    df_bi_rejeitados = (
        df.join(
            df_bi_duplicados,
            on="bi",
            how="inner"
        )
        .withColumn(
            "motivo_rejeicao",
            lit("BI duplicado")
        )
    )

    # =========================
    # Registros com NIF duplicado
    # =========================
    df_nif_rejeitados = (
        df.join(
            df_nif_duplicados,
            on="nif",
            how="inner"
        )
        .withColumn(
            "motivo_rejeicao",
            lit("NIF duplicado")
        )
    )

    # =========================
    # Registros com Email duplicado
    # =========================
    df_email_rejeitados = (
        df.join(
            df_email_duplicados,
            on="email",
            how="inner"
        )
        .withColumn(
            "motivo_rejeicao",
            lit("Email duplicado")
        )
    )

    # =========================
    # Todos os registros rejeitados
    # =========================
    df_rejeitados = (
        df_bi_rejeitados
        .unionByName(df_nif_rejeitados, allowMissingColumns=True)
        .unionByName(df_email_rejeitados, allowMissingColumns=True)
        .dropDuplicates(["id"])
    )
    #df_rejeitados.groupBy("motivo_rejeicao").count().show()
    # =========================
    # Registros válidos
    # =========================
    df_validos = (
        df.join(
            df_rejeitados.select("id"),
            on="id",
            how="left_anti"
        )
    )
    #print("Válidos:", df_validos.count())
    #print("Rejeitados:", df_rejeitados.count())
    #print("Total:", df_validos.count() + df_rejeitados.count())
    return df_validos

###**Salvar delta na silver

In [0]:
def salvar_silver(df: DataFrame, tabela: str):

    try:

        if not spark.catalog.tableExists(tabela):

            (
                df.write
                .format("delta")
                .mode("append")
                .saveAsTable(tabela)
            )

        else:

            deltaTable = DeltaTable.forName(spark, tabela)

            (
                deltaTable.alias("destino")
                .merge(
                    df.alias("origem"),
                    "destino.id = origem.id"
                )
                .whenNotMatchedInsertAll()
                .whenMatchedUpdateAll()
                .execute()
            )

    except Exception as e:

        print(f"Erro ao salvar tabela {tabela}: {e}")
        return False

    return True

In [0]:
df_cliente = spark.read.table("dbw_banco_orion.bronze.clientes")
df_cliente = normalizar_strings(df_cliente)
df_dados = padronizar_dados(df_cliente)
df_nulls = tratar_nullos(df_dados)
df_duplicados = tratar_duplicados(df_nulls)
df_valido = df_duplicados
salvar_silver(df_valido, "dbw_banco_orion.silver.clientes_tratados")
display(df_valido)
